# Part 11: GNN Virtual Screening of COCONUT Database

Uses trained GNN models (Part 9 GCN + Part 10 GIN) to screen ~154k natural products
from the COCONUT database for MDM2 inhibitory activity.

**Pipeline:** COCONUT SMILES -> graph featurization -> GNN predict -> filter hits

In [ ]:
!git clone https://github.com/arjunpahi/Natural_MDM2_Inhibitor_Discovery_using_ML.git
%cd Natural_MDM2_Inhibitor_Discovery_using_ML

In [ ]:
!pip install torch-geometric rdkit

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool, global_max_pool
from rdkit import Chem
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Define Model Architectures (same as Part 9 & 10)

In [ ]:
class GCN(nn.Module):
    def __init__(self, num_node_features=78, hidden_dim=128, num_classes=2, dropout=0.2):
        super().__init__()
        self.conv1 = GCNConv(num_node_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.bn3 = nn.BatchNorm1d(hidden_dim)
        self.dropout = dropout
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, num_classes)
        )

    def forward(self, x, edge_index, batch):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.bn3(self.conv3(x, edge_index)))
        x = torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=1)
        return self.classifier(x)


class GINEncoder(nn.Module):
    def __init__(self, num_node_features=78, hidden_dim=300, num_layers=5, dropout=0.2):
        super().__init__()
        self.num_layers = num_layers
        self.dropout = dropout
        self.gin_layers = nn.ModuleList()
        self.bn_layers = nn.ModuleList()
        self.gin_layers.append(nn.Linear(num_node_features, hidden_dim))
        self.bn_layers.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(num_layers - 1):
            self.gin_layers.append(nn.Linear(hidden_dim, hidden_dim))
            self.bn_layers.append(nn.BatchNorm1d(hidden_dim))
        self.eps = nn.ParameterList([nn.Parameter(torch.zeros(1)) for _ in range(num_layers)])

    def forward(self, x, edge_index, batch):
        for i in range(self.num_layers):
            neighbor_sum = self._aggregate(x, edge_index)
            x = self.gin_layers[i]((1 + self.eps[i]) * x + neighbor_sum)
            x = self.bn_layers[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=1)

    def _aggregate(self, x, edge_index):
        src, dst = edge_index
        messages = x[src]
        agg = torch.zeros_like(x)
        agg.scatter_add_(0, dst.unsqueeze(1).expand_as(messages), messages)
        return agg


class MDM2Classifier(nn.Module):
    def __init__(self, encoder, hidden_dim=600, num_classes=2):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, num_classes)
        )

    def forward(self, x, edge_index, batch):
        return self.classifier(self.encoder(x, edge_index, batch))

## 2. Load Trained Models

In [ ]:
gcn_model = GCN(num_node_features=78, hidden_dim=128, num_classes=2, dropout=0.2).to(device)
gcn_model.load_state_dict(torch.load('gcn_scratch_model.pth', map_location=device))
gcn_model.eval()
print("GCN (Part 9) loaded.")

gin_encoder = GINEncoder(num_node_features=78, hidden_dim=300, num_layers=5, dropout=0.2)
gin_model = MDM2Classifier(gin_encoder, hidden_dim=600, num_classes=2).to(device)
gin_model.load_state_dict(torch.load('pretrained_gin_mdm2.pth', map_location=device))
gin_model.eval()
print("GIN (Part 10) loaded.")

## 3. Load COCONUT Data

In [ ]:
coconut = pd.read_csv('Part_8/screening_results.csv')
print(f"COCONUT compounds: {len(coconut)}")
coconut.head()

In [ ]:
# OPTION B — screen directly from the raw COCONUT database (paper Section 2.5).
# Use this INSTEAD of cell 9's Part_8 file: loads coconut_csv-03-2025.csv,
# runs the paper's sequential filters (valid SMILES -> PAINS -> Brenk -> Ro5,
# zero violations like Part 7), and replaces `coconut` so all cells below
# (featurize -> GCN/GIN -> results) run on it unchanged. Skip this cell to
# keep the default Part_8 input.
import os
from rdkit.Chem import FilterCatalog, Descriptors
RAW_COCONUT = 'coconut_csv-03-2025.csv'
if not os.path.exists(RAW_COCONUT):
    print(f'{RAW_COCONUT} not found. Run the download cell in Section 11 first,')
    print('or keep the Part_8 table loaded above. Nothing changed.')
else:
    raw = pd.read_csv(RAW_COCONUT, usecols=lambda c: c in (
        'identifier', 'id', 'ID', 'name', 'canonical_smiles', 'smiles', 'SMILES'))
    smi_col = next((c for c in ['canonical_smiles', 'smiles', 'SMILES'] if c in raw.columns), None)
    id_col = next((c for c in ['identifier', 'id', 'ID', 'name'] if c in raw.columns), None)
    print(f'Raw COCONUT rows: {len(raw)}')
    _p = FilterCatalog.FilterCatalogParams()
    _p.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
    _pcat = FilterCatalog.FilterCatalog(_p)
    _b = FilterCatalog.FilterCatalogParams()
    _b.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK)
    _bcat = FilterCatalog.FilterCatalog(_b)
    keep_id, keep_smi, mw_l, logp_l, hba_l, hbd_l = [], [], [], [], [], []
    n_inv = n_pains = n_brenk = n_ro5 = 0
    for _, r in tqdm(raw.iterrows(), total=len(raw), desc='COCONUT filtering'):
        mol = Chem.MolFromSmiles(r[smi_col])
        if mol is None:
            n_inv += 1
            continue
        if _pcat.HasMatch(mol):
            n_pains += 1
            continue
        if _bcat.HasMatch(mol):
            n_brenk += 1
            continue
        mw, hba, hbd, lp = (Descriptors.ExactMolWt(mol), Descriptors.NumHAcceptors(mol),
                            Descriptors.NumHDonors(mol), Descriptors.MolLogP(mol))
        if not (mw <= 500 and hba <= 10 and hbd <= 5 and lp <= 5):
            n_ro5 += 1
            continue
        keep_id.append(r[id_col] if id_col else f'CNP{len(keep_id)}')
        keep_smi.append(r[smi_col])
        mw_l.append(mw); logp_l.append(lp); hba_l.append(hba); hbd_l.append(hbd)
    coconut = pd.DataFrame({'identifier': keep_id, 'canonical_smiles': keep_smi,
                            'mw': mw_l, 'logp': logp_l, 'n_hba': hba_l, 'n_hbd': hbd_l})
    print(f'Removed: {n_inv} invalid, {n_pains} PAINS, {n_brenk} Brenk, {n_ro5} Ro5-violating')
    print(f'Screenable COCONUT compounds: {len(coconut)}')
    coconut.head()


## 4. Featurize COCONUT SMILES -> Graphs

In [ ]:
ATOM_CHOICES = {
    'atomic_num': list(range(1, 101)),
    'degree': [0, 1, 2, 3, 4, 5],
    'formal_charge': [-2, -1, 0, 1, 2, 3],
    'num_hs': [0, 1, 2, 3, 4],
    'hybridization': [
        Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2,
        Chem.rdchem.HybridizationType.SP3, Chem.rdchem.HybridizationType.SP3D,
        Chem.rdchem.HybridizationType.SP3D2
    ]
}

def one_hot(val, choices):
    enc = [0] * len(choices)
    if val in choices:
        enc[choices.index(val)] = 1
    return enc

def atom_features(atom):
    f = []
    f += one_hot(atom.GetAtomicNum(), ATOM_CHOICES['atomic_num'])
    f += one_hot(atom.GetTotalDegree(), ATOM_CHOICES['degree'])
    f += one_hot(atom.GetFormalCharge(), ATOM_CHOICES['formal_charge'])
    f += one_hot(atom.GetTotalNumHs(), ATOM_CHOICES['num_hs'])
    f += one_hot(atom.GetHybridization(), ATOM_CHOICES['hybridization'])
    f.append(int(atom.GetIsAromatic()))
    f.append(int(atom.IsInRing()))
    return (f + [0] * 78)[:78]

def mol_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    node_features = [atom_features(a) for a in mol.GetAtoms()]
    edge_index = []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        edge_index.extend([[i, j], [j, i]])
    if not edge_index:
        edge_index = [[0, 0]]
    return Data(
        x=torch.tensor(node_features, dtype=torch.float),
        edge_index=torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    )

In [ ]:
graphs = []
valid_idx = []
failed = 0

for idx, row in tqdm(coconut.iterrows(), total=len(coconut), desc="Featurizing"):
    g = mol_to_graph(row['canonical_smiles'])
    if g is not None:
        graphs.append(g)
        valid_idx.append(idx)
    else:
        failed += 1

print(f"Converted: {len(graphs)}/{len(coconut)} | Failed: {failed}")

In [ ]:
screen_loader = DataLoader(graphs, batch_size=256, shuffle=False)
print(f"Batches: {len(screen_loader)}")

## 5. Screen with GCN (Part 9)

In [ ]:
gcn_preds, gcn_probs = [], []

gcn_model.eval()
with torch.no_grad():
    for batch in tqdm(screen_loader, desc="GCN Screening"):
        batch = batch.to(device)
        out = gcn_model(batch.x, batch.edge_index, batch.batch)
        probs = F.softmax(out, dim=1)
        gcn_probs.extend(probs[:, 1].cpu().numpy())
        gcn_preds.extend(out.argmax(dim=1).cpu().numpy())

gcn_preds = np.array(gcn_preds)
gcn_probs = np.array(gcn_probs)
print(f"GCN screening done. Predicted active: {(gcn_preds == 1).sum()}")

## 6. Screen with GIN (Part 10)

In [ ]:
gin_preds, gin_probs = [], []

gin_model.eval()
with torch.no_grad():
    for batch in tqdm(screen_loader, desc="GIN Screening"):
        batch = batch.to(device)
        out = gin_model(batch.x, batch.edge_index, batch.batch)
        probs = F.softmax(out, dim=1)
        gin_probs.extend(probs[:, 1].cpu().numpy())
        gin_preds.extend(out.argmax(dim=1).cpu().numpy())

gin_preds = np.array(gin_preds)
gin_probs = np.array(gin_probs)
print(f"GIN screening done. Predicted active: {(gin_preds == 1).sum()}")

## 7. Assemble Results

In [ ]:
# Build results DataFrame for valid molecules only
results = coconut.iloc[valid_idx][['identifier', 'canonical_smiles']].copy()

# Add RF predictions from Part 8 (optional baseline — skipped when screening raw COCONUT)
try:
    rf_orig = pd.read_csv('Part_8/screening_results.csv')
    if set(['prediction', 'prob_class_1']).issubset(rf_orig.columns) and len(rf_orig) == len(coconut):
        results['rf_prediction'] = rf_orig.iloc[valid_idx]['prediction'].values
        results['rf_prob_active'] = rf_orig.iloc[valid_idx]['prob_class_1'].values
        HAVE_RF = True
    else:
        raise ValueError('Part 8 file does not match the current coconut table')
except Exception as e:
    print(f'RF baseline skipped ({e}). Consensus will use GCN + GIN only.')
    HAVE_RF = False

# Add GNN predictions
results['gcn_prediction'] = gcn_preds
results['gcn_prob_active'] = gcn_probs
results['gin_prediction'] = gin_preds
results['gin_prob_active'] = gin_probs

if HAVE_RF:
    # Consensus: active if majority of 3 models predict active
    results['consensus'] = ((results['rf_prediction'] + results['gcn_prediction'] + results['gin_prediction']) >= 2).astype(int)
else:
    # Consensus: both GNN models predict active
    results['consensus'] = ((results['gcn_prediction'] + results['gin_prediction']) == 2).astype(int)

print(f"Total screened: {len(results)}")
print(f"\nPredicted active by model:")
if HAVE_RF:
    print(f"  RF:   {results['rf_prediction'].sum()}")
print(f"  GCN:  {results['gcn_prediction'].sum()}")
print(f"  GIN:  {results['gin_prediction'].sum()}")
print(f"  Consensus: {results['consensus'].sum()}")
results.head(10)


In [ ]:
results.to_csv('gnn_screening_results.csv', index=False)

In [ ]:
print('Saved: gnn_screening_results.csv')

## 8. Filter High-Confidence Hits

In [ ]:
# Consensus hits (2/3 models agree)
consensus_hits = results[results['consensus'] == 1].copy()
consensus_hits = consensus_hits.sort_values('gin_prob_active', ascending=False)
print(f"Consensus hits: {len(consensus_hits)}")
consensus_hits

In [ ]:
consensus_hits.to_csv('gnn_consensus_hits.csv', index=False)

In [ ]:
# GIN-only high confidence (prob > 0.6)
gin_high_conf = results[(results['gin_prediction'] == 1) & (results['gin_prob_active'] > 0.6)].copy()
gin_high_conf = gin_high_conf.sort_values('gin_prob_active', ascending=False)
print(f"GIN high-confidence hits (>0.6): {len(gin_high_conf)}")
gin_high_conf

## 9. Compare GNN vs RF Screening

In [ ]:
if 'rf_prediction' in results.columns:
    # How many compounds do all 3 models agree on?
    all_agree = results[(results['rf_prediction'] == 1) & (results['gcn_prediction'] == 1) & (results['gin_prediction'] == 1)]
    print(f"All 3 models agree (active): {len(all_agree)}")

    # Only GNN finds (not RF)
    gnn_only = results[(results['gcn_prediction'] == 1) | (results['gin_prediction'] == 1) & (results['rf_prediction'] == 0)]
    gnn_only_unique = gnn_only[~gnn_only.index.isin(results[results['rf_prediction'] == 1].index)]
    print(f"GNN-only finds (not RF): {len(gnn_only_unique)}")

    # Only RF finds (not GNN)
    rf_only = results[(results['rf_prediction'] == 1) & (results['gcn_prediction'] == 0) & (results['gin_prediction'] == 0)]
    print(f"RF-only finds (not GNN): {len(rf_only)}")
else:
    # Raw-COCONUT mode: GNN-vs-GNN agreement only (no RF baseline available)
    both = results[(results['gcn_prediction'] == 1) & (results['gin_prediction'] == 1)]
    print(f"GCN+GIN agree (active): {len(both)}")
    print(f"GCN-only: {len(results[(results['gcn_prediction'] == 1) & (results['gin_prediction'] == 0)])}")
    print(f"GIN-only: {len(results[(results['gcn_prediction'] == 0) & (results['gin_prediction'] == 1)])}")


In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(8, 5))
categories = ['RF only', 'GCN only', 'GIN only', 'RF+GCN', 'RF+GIN', 'GCN+GIN', 'All 3']
counts = [
    len(rf_only),
    len(results[(results['gcn_prediction']==1) & (results['rf_prediction']==0) & (results['gin_prediction']==0)]),
    len(results[(results['gin_prediction']==1) & (results['rf_prediction']==0) & (results['gcn_prediction']==0)]),
    len(results[(results['rf_prediction']==1) & (results['gcn_prediction']==1) & (results['gin_prediction']==0)]),
    len(results[(results['rf_prediction']==1) & (results['gin_prediction']==1) & (results['gcn_prediction']==0)]),
    len(results[(results['gcn_prediction']==1) & (results['gin_prediction']==1) & (results['rf_prediction']==0)]),
    len(all_agree)
]
colors = ['steelblue', 'coral', 'green', 'mediumpurple', 'orange', 'teal', 'gold']
ax.bar(categories, counts, color=colors, edgecolor='black')
ax.set_ylabel('Number of Compounds')
ax.set_title('Model Agreement on COCONUT Screening')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('gnn_vs_rf_screening.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. Save SMILES for Downstream Docking

In [ ]:
# Save consensus hit SMILES
with open('gnn_consensus_smiles.txt', 'w') as f:
    for smi in consensus_hits['canonical_smiles']:
        f.write(smi + '\n')
print(f"Saved {len(consensus_hits)} SMILES to gnn_consensus_smiles.txt")

In [ ]:
print("\n=== Part 11 Complete ===")
print("Outputs:")
print("  - gnn_screening_results.csv (all compounds with RF/GCN/GIN predictions)")
print("  - gnn_consensus_hits.csv (2/3 models agree = active)")
print("  - gnn_consensus_smiles.txt (SMILES for docking)")
print("  - gnn_vs_rf_screening.png (comparison plot)")

## 11. Paper-style pipeline: COCONUT download + PAINS/Brenk/Ro5 + deep-learning screening

Follows the paper (Sections 2.5 and 3.5). The ML implementation downloads COCONUT
(`coconut_csv-03-2025.csv`), applies sequential PAINS → Brenk → Lipinski filtering,
screens with the Random Forest model and keeps `prob > 0.6` (116 hits) for docking.

Below is the **same pipeline in deep-learning mode**: identical filters, but GCN + GIN
probabilities instead of RF. It reuses `results` built in the cells above,
so run the notebook top-to-bottom first.


In [ ]:
import os
COCONUT_CSV = 'coconut_csv-03-2025.csv'  # paper Data Availability snapshot, March 2025
FILE_ID = '1-DFc6lMf6maNAWPwZFA8uY6951SokoNY'  # same public Drive file as Part 7
if os.path.exists(COCONUT_CSV):
    print(f'Found {COCONUT_CSV}, download skipped.')
else:
    try:
        import gdown
    except ImportError:
        %pip install gdown -q
        import gdown
    try:
        gdown.download(f'https://drive.google.com/uc?id={FILE_ID}', COCONUT_CSV, quiet=False)
        print(f'Downloaded {COCONUT_CSV}')
    except Exception as e:
        print(f'Download failed ({e}).')
        print('Falling back to Part_8/screening_results.csv already loaded above.')


In [ ]:
from rdkit.Chem import FilterCatalog, Descriptors

# Sequential med-chem filters, same as paper Section 2.5 / Part 7
_p = FilterCatalog.FilterCatalogParams()
_p.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
PAINS_CAT = FilterCatalog.FilterCatalog(_p)
_b = FilterCatalog.FilterCatalogParams()
_b.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK)
BRENK_CAT = FilterCatalog.FilterCatalog(_b)

def medchem_pass(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False, False, False
    pains_ok = not PAINS_CAT.HasMatch(mol)
    brenk_ok = not BRENK_CAT.HasMatch(mol)
    ro5_ok = (Descriptors.ExactMolWt(mol) <= 500 and Descriptors.NumHAcceptors(mol) <= 10
              and Descriptors.NumHDonors(mol) <= 5 and Descriptors.MolLogP(mol) <= 5)
    return pains_ok, brenk_ok, ro5_ok

if 'results' not in dir():
    print("Run the cells above first so that 'results' exists.")
else:
    flags = [medchem_pass(s) for s in results['canonical_smiles']]
    results['pains_pass'] = [f[0] for f in flags]
    results['brenk_pass'] = [f[1] for f in flags]
    if 'ro5_pass' not in results.columns:
        results['ro5_pass'] = [f[2] for f in flags]
    n = len(results)
    print(f'PAINS pass: {results["pains_pass"].sum()} / {n}')
    print(f'Brenk pass: {results["brenk_pass"].sum()} / {n}')
    print(f'Ro5 pass:   {results["ro5_pass"].sum()} / {n}')
    print(f'All three:  {(results["pains_pass"] & results["brenk_pass"] & results["ro5_pass"]).sum()} / {n}')


In [ ]:
# Deep-learning screening mode (paper Section 3.5 logic, GNN probabilities).
# Paper ML rule: RF prob_class_1 > 0.6 -> 116 hits for docking.
# DL rule below: mean(GCN prob, GIN prob) > cutoff, plus med-chem filters.
GCN_COL, GIN_COL = 'gcn_prob_active', 'gin_prob_active'
if 'results' not in dir():
    print("Run the cells above first so that 'results' exists.")
else:
    results['avg_prob'] = (results[GCN_COL] + results[GIN_COL]) / 2
    med = pd.Series(True, index=results.index)
    for c in ['pains_pass', 'brenk_pass', 'ro5_pass']:
        if c in results.columns:
            med &= results[c]
    print('Threshold sweep (with med-chem filters where available):')
    for t in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9]:
        m = (results['avg_prob'] > t) & med
        print(f'  avg_prob > {t}: {m.sum()} hits')
    PAPER_CUT = 0.6  # same cutoff value as the paper's RF prob > 0.6
    dl_hits = results[(results['avg_prob'] > PAPER_CUT) & med].sort_values('avg_prob', ascending=False)
    keep = ['identifier', 'canonical_smiles', GCN_COL, GIN_COL, 'avg_prob']
    dl_hits[keep].to_csv('gnn_paperstyle_hits.csv', index=False)
    print(f'\nSaved {len(dl_hits)} DL hits (avg_prob > {PAPER_CUT}) to gnn_paperstyle_hits.csv')
    try:
        rf116 = pd.read_csv('Part_8/filtered_compounds.csv')
        print(f"Paper RF hits for comparison: {len(rf116)} (prob > 0.6)")
    except Exception:
        print('(Part_8/filtered_compounds.csv not found, skipping RF comparison.)')
    dl_hits[keep].head(10)


In [ ]:
# SDF extraction of DL hits for docking (paper Sections 2.6/3.6, mirrors Part 8).
# Set SDF_PATH to your local COCONUT SDF file, then run this cell.
import os
SDF_PATH = ''  # e.g. 'coconut_sdf_3d-04-2025.sdf'
SRC = 'dl_hits' if 'dl_hits' in dir() else ('hits' if 'hits' in dir() else ('consensus_hits' if 'consensus_hits' in dir() else None))
if SRC is None:
    print('No hits table found yet. Run the screening cells above first.')
elif not SDF_PATH or not os.path.exists(SDF_PATH):
    print('Set SDF_PATH to the COCONUT SDF file to extract 3D structures for docking.')
    print(f'(Would extract {len(eval(SRC))} compounds from {SRC}.)')
else:
    target_ids = set(eval(SRC)['identifier'].dropna().astype(str))
    supplier = Chem.SDMolSupplier(SDF_PATH)
    writer = Chem.SDWriter('gnn_hits_for_docking.sdf')
    count = 0
    for mol in supplier:
        if mol is None:
            continue
        try:
            if mol.GetProp('IDENTIFIER') in target_ids:
                writer.write(mol)
                count += 1
        except KeyError:
            continue
    writer.close()
    print(f'Extracted {count} DL-hit structures to gnn_hits_for_docking.sdf')
